# Notas de Estudio y Desarrollo: Optimización Cuántica

Este cuaderno contiene apuntes teóricos, desarrollos matemáticos e implementaciones prácticas basados en los capítulos de optimización cuántica del libro:
* **Manual:** *A Practical Guide to Quantum Machine Learning and Quantum Optimization* (Packt Publishing).
* **Autores:** Elías F. Combarro y Samuel González-Castillo.

### Contenido y Alcance
* **Fundamentos de PennyLane:** Configuración de QNodes, parametrización de circuitos y gestión de tipos de mediciones (probabilidades y muestreos analíticos).
* **Modelado de Problemas Combinatorios:** Traducción matemática de problemas NP-difíciles (como Max-Cut) al marco de optimización binaria cuadrática sin restricciones (**QUBO**) y su equivalencia con el **modelo de Ising**.
* **Simulación de Recocido Cuántico:** Implementación y resolución del estado fundamental de energía utilizando el simulador local de *quantum annealing* (`dwave-neal`).

## 2.3.1 Circuit engineering 101

La forma en que se construyen los circuitos cuánticos en PennyLane es fundamentalmente diferente a la de Qiskit. En PennyLane, los circuitos se representan como **nodos cuánticos (QNodes)**, los cuales encapsulan una función cuántica (la especificación del circuito) y un dispositivo (donde se ejecutará).

El primer paso es siempre crear el dispositivo utilizando `qml.device()`, indicando el nombre del backend (por ejemplo, el simulador `default.qubit`) y el número de qubits (llamados *wires* o cables).


In [15]:
import pennylane as qml
import numpy as np

# Definimos un dispositivo con 2 qubits
dev = qml.device('default.qubit', wires = 2)

### Definición y Ensamblaje Manual del Circuito

La especificación del circuito se define como una función normal de Python. Dentro de ella, aplicamos las puertas indicando explícitamente los `wires` (cables) sobre los que actúan. La salida de la función debe ser la información que queremos obtener, como por ejemplo el vector de estado usando `qml.state()`.

Para ejecutarlo, ensamblamos la función y el dispositivo usando `qml.QNode()`.


In [16]:
# 1. Definimos la función del circuito
def qc():
    qml.PauliX(wires = 0)
    qml.Hadamard(wires = 0)
    return qml.state()

# 2. Ensamblamos la función y el dispositivo en un QNode
qcirc = qml.QNode(qc, dev)

# 3. Ejecutamos
estado = qcirc()
print("Vector de estado:", estado)

Vector de estado: [ 0.70710678+0.j  0.        +0.j -0.70710678+0.j  0.        +0.j]


### El atajo del Decorador y Dibujo de Circuitos

Ensamblar circuitos manualmente puede ser tedioso. PennyLane ofrece un atajo usando el decorador `@qml.qnode(dev)` justo antes de definir la función del circuito.

Al ser funciones nativas de Python, podemos **parametrizar el circuito pasando argumentos** (como ángulos) e incluso usar lógica clásica (bucles `for` o condicionales `if`) directamente dentro de la función. 

Para visualizar el circuito, utilizamos la función `qml.draw()`.


In [17]:
# 1. Definimos un dispositivo de 1 qubit
dev1 = qml.device('default.qubit', wires = 1)

# 2. Usamos el decorador para unir función y dispositivo
@qml.qnode(dev1)
def qcirc_parametrizado(theta):
    qml.RX(theta, wires = 0) # ¡Aquí sí pasamos el wire correctamente!
    return qml.state()

# 3. Ejecutamos pasándole un parámetro
print("Estado con theta=2.0:\n", qcirc_parametrizado(2.0))

# 4. Dibujamos el circuito
print("\nDibujo del circuito:")
print(qml.draw(qcirc_parametrizado)(theta = 2.0))

Estado con theta=2.0:
 [0.54030231+0.j         0.        -0.84147098j]

Dibujo del circuito:
0: ──RX(2.00)─┤  State


### Tipos de Mediciones (Returns)

Hasta ahora solo hemos devuelto el vector de estado exacto (`qml.state()`), pero PennyLane ofrece más opciones:
* `qml.probs(wires=...)`: Devuelve una lista con las probabilidades de cada estado de la base computacional.
* `qml.sample(wires=...)`: Devuelve muestras individuales de mediciones. Para usar esto, es necesario haber especificado el número de `shots` al inicializar el dispositivo o al ejecutar el QNode.


In [18]:
# --- Ejemplo 1: Probabilidades ---
dev3 = qml.device('default.qubit', wires = 3)

@qml.qnode(dev3)
def qcirc_probs():
    qml.Hadamard(wires = 1)
    # Pedimos probabilidades SOLO para los cables 1 y 2
    return qml.probs(wires = [1, 2]) 

print("Probabilidades cables [1, 2]:", qcirc_probs())

# --- Ejemplo 2: Muestreo (Samples) indicando shots en el QNode ---
@qml.qnode(dev3, shots = 4)  # <-- Declaramos los 4 shots aquí dentro
def qcirc_sample1():
    qml.Hadamard(wires = 0)
    return qml.sample(wires = 0) 

# Ejecutamos la función limpia, sin pasarle argumentos obsoletos
print("\nMuestra 1 (4 shots):", qcirc_sample1())

# --- Ejemplo 3: Muestreo (Samples) indicando shots en el dispositivo ---
dev_shots = qml.device('default.qubit', wires = 2, shots = 4)

@qml.qnode(dev_shots)
def qcirc_sample2():
    qml.Hadamard(wires=0)
    # Al no pasar 'wires', muestreará todos los cables
    return qml.sample() 

print("\nMuestra 2:", qcirc_sample2())


Probabilidades cables [1, 2]: [0.5 0.  0.5 0. ]

Muestra 1 (4 shots): [[1]
 [0]
 [0]
 [1]]

Muestra 2: [[0 0]
 [0 0]
 [1 0]
 [1 0]]


## 2.3.2 Interoperabilidad de PennyLane

Una de las características más potentes de PennyLane es su capacidad de comunicarse con otros frameworks cuánticos. Al instalar el plugin de Qiskit (`pennylane-qiskit`), obtenemos acceso a una nueva serie de dispositivos (devices). 

Entre ellos destaca `qiskit.aer`, el cual nos permite utilizar el simulador de alto rendimiento Aer de Qiskit directamente desde PennyLane.

### Ejecutando circuitos en Qiskit Aer (Love is in the Aer)
Para simular un circuito en PennyLane usando Aer, lo único que debemos cambiar es el dispositivo al momento de inicializarlo, pasando el string `'qiskit.aer'` en lugar de `'default.qubit'`.

In [19]:
import pennylane as qml

# 1. Definimos el dispositivo usando el simulador qiskit.aer
dev = qml.device('qiskit.aer', wires = 2)

# 2. Creamos el QNode y el circuito
@qml.qnode(dev)
def qcirc():
    qml.Hadamard(wires = 0)
    return qml.probs(wires = 0)

# 3. Ejecutamos
s = qcirc()
print("Las probabilidades son:", s)


Las probabilidades son: [0.49804688 0.50195312]


### Obteniendo el Vector de Estado Exacto

Al ejecutar el código anterior, obtendremos probabilidades empíricas (es decir, extraídas a partir de un número de muestras o *shots* limitados, por lo que no serán exactamente 0.5 y 0.5). 

Si lo que deseamos es obtener el vector de estado analítico exacto (ideal), debemos indicarle al dispositivo que utilice específicamente el backend de simulación de vectores de estado (`aer_simulator_statevector`) y establecer los `shots` a `None`. Al hacer esto, también podremos usar `qml.state()` y `qml.probs()` devolverá resultados analíticos perfectos.


In [20]:
# 1. Definimos el dispositivo especificando el backend y eliminando los shots
dev_exacto = qml.device('qiskit.aer', 
                        wires = 2,
                        backend='aer_simulator_statevector', 
                        shots = None)

# 2. Creamos el QNode
@qml.qnode(dev_exacto)
def qcirc_exacto():
    qml.Hadamard(wires = 0)
    # Ahora podemos devolver el estado matemático exacto
    return qml.state()

# 3. Ejecutamos
estado = qcirc_exacto()
print("El vector de estado exacto es:\n", estado)


El vector de estado exacto es:
 [0.70710678+0.j 0.        +0.j 0.70710678+0.j 0.        +0.j]


# Capítulo 3: Problemas de Optimización Binaria Cuadrática Sin Restricciones (QUBO)

A partir de ahora, vamos a estudiar algoritmos cuánticos diseñados para resolver problemas de optimización, como el **Quantum Approximate Optimization Algorithm (QAOA)**, el **Grover Adaptive Search (GAS)** y el **Variational Quantum Eigensolver (VQE)**. 

Sin embargo, antes de poder usar un ordenador cuántico para resolver un problema de la vida real, necesitamos traducir ese problema a un "idioma" que la máquina cuántica pueda entender. Ese lenguaje matemático es el marco **QUBO** (Optimización Binaria Cuadrática Sin Restricciones) y el **modelo de Ising**. 


## 3.1 El problema Max-Cut y el modelo de Ising

Para entender cómo formular problemas para un ordenador cuántico, vamos a utilizar como ejemplo principal uno de los problemas clásicos más famosos de la computación: el problema del **Max-Cut** (Corte Máximo).

### 3.1.1 Grafos y cortes (Graphs and cuts)

Para entender el problema Max-Cut, primero debemos definir sus dos elementos principales:
1. **Un Grafo:** Es una estructura matemática formada por puntos llamados **vértices** (o nodos) y conexiones entre pares de estos puntos llamadas **aristas**.
2. **Un Corte:** Consiste en tomar todos los vértices del grafo y dividirlos exactamente en **dos conjuntos separados** (por ejemplo, el conjunto A y el conjunto B).

**¿Qué es el problema Max-Cut?**
Cada vez que dividimos los vértices en dos conjuntos, algunas aristas conectarán dos vértices que han quedado en el mismo conjunto, y otras aristas conectarán un vértice del conjunto A con un vértice del conjunto B. A estas últimas las llamamos "aristas cortadas". 

El **problema Max-Cut** consiste en encontrar la forma óptima de dividir los vértices para que **el número de aristas que cruzan de un conjunto a otro sea el máximo posible**. En otras palabras, queremos maximizar las conexiones que tienen sus extremos en grupos diferentes.

*   *Ejemplo intuitivo:* Si un corte "rompe" 4 aristas, decimos que ese corte tiene un tamaño de 4. Nuestro objetivo es encontrar el corte que tenga el tamaño máximo.

### 3.1.2 Formulando el problema (Formulating the problem)

Para que un ordenador pueda resolver el problema del Max-Cut, necesitamos una formulación matemática sin hacer referencia a dibujos de grafos, vértices o cuerdas.

Para lograr esto, el primer paso es **asignar una variable $z_i$ a cada vértice** $i$ del grafo. La clave de esta formulación es que estas variables son binarias, pero en lugar de usar 0 y 1, **solo pueden tomar los valores $1$ o $-1$**.

Cada posible asignación de valores a estas variables determina automáticamente un corte en el grafo:
*   Los vértices cuya variable $z_i$ tome el valor $1$ estarán en un conjunto (por ejemplo, el Conjunto A).
*   Los vértices cuya variable $z_i$ tome el valor $-1$ estarán en el otro conjunto (Conjunto B).


#### La función objetivo a minimizar

Una vez que hemos asignado un valor ($1$ o $-1$) a cada vértice, ¿cómo sabemos matemáticamente si la arista que conecta el vértice $j$ con el vértice $k$ ha sido cortada? Simplemente multiplicamos sus variables ($z_j \cdot z_k$). Solo pueden darse dos casos:

1. **Los vértices están en el mismo conjunto:** Sus variables son iguales, por lo que el producto será $(1 \cdot 1 = 1)$ o $(-1 \cdot -1 = 1)$. La arista **no está cortada**.
2. **Los vértices están en conjuntos diferentes:** Sus variables son opuestas, por lo que el producto será $(1 \cdot -1 = -1)$ o $(-1 \cdot 1 = -1)$. La arista **sí está cortada**.

Como las aristas cortadas aportan un valor de $-1$, encontrar el corte máximo (Max-Cut) es matemáticamente equivalente a **minimizar la suma de los productos de todas las aristas**. 

Si llamamos $E$ al conjunto de todas las aristas del grafo, nuestro problema se formula así:

*   **Minimizar:**  $\sum_{(j,k) \in E} z_j z_k$
*   **Sujeto a:**  $z_j \in \{-1, 1\}$ para todo $j$.


#### La complejidad del problema Max-Cut

A simple vista, minimizar la suma anterior podría parecer una tarea sencilla. Sin embargo, el Max-Cut es un famoso problema **NP-hard** (NP-difícil). 

Esto significa que, si fuéramos capaces de resolver este problema de manera eficiente con un algoritmo clásico, se demostraría que $P = NP$, algo que la comunidad científica cree firmemente que es falso. Incluso intentar buscar aproximaciones suficientemente buenas mediante métodos clásicos es extremadamente difícil. Es por este motivo que resulta tan interesante intentar resolver (o aproximar) el Max-Cut utilizando ordenadores cuánticos.


Copia este texto para introducir la conexión entre el Max-Cut y la física estadística.
### 3.1.3 El modelo de Ising (The Ising model)

El problema del Max-Cut, tal y como lo hemos formulado matemáticamente, puede verse como un caso particular de un problema aparentemente no relacionado en el campo de la física estadística: **encontrar el estado de mínima energía de una instancia del modelo de Ising**.

Para los físicos, el modelo de Ising es un modelo matemático que describe la interacción ferromagnética de partículas con espín (dispuestas normalmente en una red geométrica). En este modelo, el espín de cada partícula se representa mediante una variable $z_j$ que puede tomar los valores $1$ (espín hacia arriba) o $-1$ (espín hacia abajo). ¡Exactamente las mismas variables binarias que usamos para los vértices en el problema Max-Cut!.


#### La función Hamiltoniana del modelo de Ising

En física, la energía total de este sistema viene dada por una cantidad llamada **función Hamiltoniana**. Para el modelo de Ising, esta ecuación se define de la siguiente manera:

$$ E = - \sum_{j,k} J_{jk} z_j z_k - \sum_j h_j z_j $$

Donde:
*   **$J_{jk}$**: Son coeficientes que representan la fuerza de interacción entre las partículas $j$ y $k$ (normalmente solo es distinto de cero para partículas adyacentes).
*   **$h_j$**: Son coeficientes que representan la influencia de un campo magnético externo sobre la partícula $j$.


Encontrar el **estado de mínima energía** del sistema consiste simplemente en obtener la configuración de espines ($1$ o $-1$) para la cual esta función Hamiltoniana alcanza su valor mínimo.


#### Equivalencia entre Ising y Max-Cut

Si observamos detenidamente la función Hamiltoniana, podemos notar algo fascinante. Si configuramos todos los coeficientes del campo magnético externo a cero ($h_j = 0$) y hacemos que todos los coeficientes de interacción sean iguales a menos uno ($J_{jk} = -1$), la ecuación de la energía se simplifica a:

$$ E = \sum_{j,k} z_j z_k $$

¡Este problema es **exactamente el mismo** que encontrar el corte máximo (Max-Cut) en un grafo!. 

Dado que el Max-Cut es un problema NP-difícil, esto implica que encontrar el estado de mínima energía de un modelo de Ising dado es, por extensión, también un problema **NP-difícil** (NP-hard). La gran ventaja es que los ordenadores cuánticos (como los *quantum annealers* de D-Wave) están construidos específicamente para encontrar estos estados fundamentales de energía siguiendo las leyes de la física.


## 3.2.1 De variables clásicas a qubits (From classical variables to qubits)

Hasta ahora, las formulaciones para el problema de Max-Cut y el modelo de Ising han sido puramente clásicas. Utilizaban variables $z_j$ que solo podían tomar los valores $1$ o $-1$. 

Para llevar esto a un ordenador cuántico, el "truco" consiste en sustituir esas variables matemáticas por **operadores cuánticos**, concretamente por la **puerta de Pauli $Z$** actuando sobre los estados de la base computacional $|0\rangle$ y $|1\rangle$.

Si calculamos el valor esperado de la puerta $Z$ para estos estados, obtenemos exactamente los mismos valores clásicos del modelo de Ising:
* $\langle 0 | Z | 0 \rangle = 1$
* $\langle 1 | Z | 1 \rangle = -1$

### Evaluando aristas con productos tensoriales

Para saber si una arista que conecta los vértices $j$ y $k$ ha sido cortada bajo una asignación de bits $x$, evaluamos el producto tensorial de sus puertas: $\langle x | Z_j Z_k | x \rangle$. 

El resultado de esta operación seguirá exactamente la misma lógica:
* Si los qubits tienen distinto valor (ej. $|0\rangle$ y $|1\rangle$), la arista está cortada y el resultado es **$-1$**.
* Si los qubits tienen el mismo valor, la arista no está cortada y el resultado es **$1$**.


Por linealidad de las matrices, podemos sumar todas las aristas. Así, encontrar el estado de la base $|x\rangle$ que minimice la suma de estas evaluaciones equivale a encontrar el corte máximo (Max-Cut).

### Superposición y la formulación cuántica final

La verdadera ventaja de la formulación cuántica surge cuando pasamos de estados clásicos $|x\rangle$ a un estado cuántico general en superposición $|\psi\rangle = \sum_x a_x |x\rangle$.

Al calcular el valor esperado del operador para este estado general, las matemáticas nos dicen que:

$$ \langle \psi | Z_j Z_k | \psi \rangle = \sum_y \sum_x a_y^* a_x \langle y | Z_j Z_k | x \rangle = \sum_x |a_x|^2 \langle x | Z_j Z_k | x \rangle $$

Como el término $|a_x|^2$ representa la probabilidad de medir el estado $|x\rangle$ al final del circuito, minimizar este valor esperado sobre **todos los estados cuánticos posibles** nos llevará indefectiblemente al estado de la base que minimiza la energía y resuelve el problema.


### El problema de optimización cuántica

Teniendo en cuenta todo lo anterior, dado un grafo con un conjunto de aristas $E$, podemos reescribir el problema original del Max-Cut puramente en lenguaje cuántico de la siguiente forma:

* **Minimizar:** $\sum_{(j,k) \in E} \langle \psi | Z_j Z_k | \psi \rangle$
* **Sujeto a:** $|\psi\rangle$ es un estado cuántico en $n$ qubits.

Matemáticamente, la matriz creada por $\sum_{(j,k) \in E} Z_j Z_k$ es un operador Hermítico (el Hamiltoniano del sistema). Al buscar el estado $|\psi\rangle$ que minimiza este valor esperado, lo que estamos haciendo es pedirle al ordenador cuántico que encuentre el **estado fundamental (ground state)** de ese Hamiltoniano.


### 3.2.2 Computando valores esperados con Qiskit (Computing expectation values with Qiskit)

Además de construir circuitos, Qiskit nos permite manipular estados y Hamiltonianos matemáticamente. 

El primer paso es aprender a definir estados de la base computacional, como el estado $|100\rangle$, usando la clase `Statevector`. Qiskit nos ofrece varias alternativas:
1. Pasando directamente las amplitudes de los qubits individuales y usando el operador `^` para el producto tensorial.
2. Instanciándolo directamente desde un número entero usando el método `from_int`.

Podemos incluso crear superposiciones matemáticas de manera directa sumando los vectores y multiplicándolos por sus amplitudes.


In [21]:
from qiskit.quantum_info import Statevector
from numpy import sqrt
from IPython.display import display

# Inicializamos los estados base de un solo qubit
zero = Statevector([1,0])
one  = Statevector([0,1])

# --- Opción 1: Operador ^ para producto tensorial ---
# NOTA: En Qiskit, el qubit 0 es el de más a la derecha. 
# Para |100>, el orden es one ^ zero ^ zero
psi = one ^ zero ^ zero
print("Estado |100>:")
display(psi.draw("latex"))  # type: ignore

# --- Opción 2: Desde un número entero ---
# 4 en binario es 100. dims=8 indica que son 3 qubits (2^3 = 8 amplitudes)
psi_int = Statevector.from_int(4, dims = 8)

# --- Extra: Creando superposiciones directamente ---
ghz = 1/sqrt(2)*(zero^zero^zero) + 1/sqrt(2)*(one^one^one)
print("\nEstado en superposición:")
display(ghz.draw("latex"))  # type: ignore


Estado |100>:


<IPython.core.display.Latex object>


Estado en superposición:


<IPython.core.display.Latex object>

#### Definiendo el Hamiltoniano

Para el problema de Max-Cut de nuestro grafo de ejemplo (con aristas conectando los nodos 0-1 y 0-2), vimos que nuestro Hamiltoniano a minimizar era $Z_0Z_1 + Z_0Z_2$. 

En Qiskit, podemos definir estos productos tensoriales importando los operadores de Pauli `I` y `Z` del módulo `qiskit.opflow`. Utilizaremos nuevamente el operador `^` para el producto tensorial, recordando añadir la matriz Identidad ($I$) en la posición del qubit sobre el que no estamos actuando.

**⚠️ Precaución con la sintaxis:** Qiskit sobrecarga el operador `^` para que funcione como producto tensorial. Sin embargo, en Python, el símbolo `^` tiene menor prioridad matemática que el símbolo `+`. Por lo tanto, **es obligatorio usar paréntesis** en cada término antes de sumarlos, o el código fallará.


In [22]:
from qiskit.quantum_info import SparsePauliOp

# Definimos los operadores base
I = SparsePauliOp("I")
Z = SparsePauliOp("Z")

# Construimos el Hamiltoniano Z_0 Z_1 + Z_0 Z_2 
# (Recordemos: el qubit 0 está a la derecha en Qiskit, por lo que Z_0 Z_1 es ZZI)
H_cut = (Z^Z^I) + (Z^I^Z)

print("Hamiltoniano H_cut:")
print(H_cut)

# Podemos imprimir su matriz dispersa (solo elementos no nulos) 
# para comprobar que, al ser suma de matrices diagonales, es diagonal.
# El parámetro sparse=True nos devuelve una matriz dispersa de SciPy
print("\nMatriz dispersa de H_cut:")
print(H_cut.to_matrix(sparse=True))


Hamiltoniano H_cut:
SparsePauliOp(['ZZI', 'ZIZ'],
              coeffs=[1.+0.j, 1.+0.j])

Matriz dispersa de H_cut:
<Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 8 stored elements and shape (8, 8)>
  Coords	Values
  (0, 0)	(2+0j)
  (1, 1)	0j
  (2, 2)	0j
  (3, 3)	(-2+0j)
  (4, 4)	(-2+0j)
  (5, 5)	0j
  (6, 6)	0j
  (7, 7)	(2+0j)


#### Calculando el Valor Esperado (Expectation Value)

Teniendo nuestro estado cuántico en la variable `psi` (representando el corte donde el nodo 2 está separado del 0 y 1) y el Hamiltoniano en `H_cut`, calcular la energía de este corte se reduce a una sola línea de código llamando al método `expectation_value`. 

Como la matriz es un operador Hermítico, el resultado siempre será un número real (su parte imaginaria `j` será siempre $0$).


In [23]:
# Calculamos la energía del estado |100> sobre el Hamiltoniano
energia = psi.expectation_value(H_cut)

print("El valor esperado (energía del corte) es:", energia)


El valor esperado (energía del corte) es: (-2+0j)


## 3.3 De Ising a QUBO y viceversa (Moving from Ising to QUBO and back)

Hasta ahora hemos formulado los problemas utilizando el **modelo de Ising**, donde las variables $z_j$ toman los valores $1$ o $-1$ (representando los espines físicos). Sin embargo, a la hora de modelar muchos problemas clásicos de la informática (como el famoso problema de la *Suma de Subconjuntos* o *Subset Sum*), resulta mucho más natural e intuitivo pensar en **variables que toman los valores $0$ o $1$**. Por ejemplo, podemos usar un $1$ si decidimos incluir un elemento en nuestra solución y un $0$ si decidimos descartarlo.

A los problemas de optimización que utilizan estas variables binarias de $0$ y $1$ y que tienen una función de coste cuadrática (es decir, multiplicando como mucho dos variables a la vez), se les conoce como problemas **QUBO** (*Quadratic Unconstrained Binary Optimization*, u Optimización Binaria Cuadrática Sin Restricciones).

### La transformación matemática

Puesto que la única diferencia matemática entre el modelo de Ising y el modelo QUBO es el rango de valores que pueden tomar sus variables ($z_j \in \{-1, 1\}$ frente a $x_j \in \{0, 1\}$), es sumamente sencillo traducir cualquier problema de un formato a otro aplicando una simple sustitución lineal:

*   **De QUBO a Ising:** Sustituimos cada variable $x_j$ por la expresión $\frac{1 - z_j}{2}$.
*   **De Ising a QUBO:** Sustituimos cada variable $z_j$ por la expresión $1 - 2x_j$.

Al hacer estas sustituciones y expandir los productos algebraicos resultantes, un problema QUBO se convierte instantáneamente en un problema de Ising (y viceversa). Esto es crucial, ya que nos permite modelar nuestro problema en el formato que nos resulte más cómodo (normalmente QUBO) y luego traducirlo a Ising para que el ordenador cuántico lo resuelva.


### Un ejemplo práctico de transformación

Imaginemos que tenemos la siguiente energía o función objetivo en el modelo de Ising que queremos minimizar:
$$ E = -\frac{1}{2} z_0 z_1 + z_2 $$

Para transformarlo a su formato QUBO, simplemente aplicamos la sustitución $z_j = 1 - 2x_j$:
$$ E = -\frac{1}{2} (1 - 2x_0)(1 - 2x_1) + (1 - 2x_2) $$
$$ E = -\frac{1}{2} (1 - 2x_0 - 2x_1 + 4x_0x_1) + 1 - 2x_2 $$
$$ E = -\frac{1}{2} + x_0 + x_1 - 2x_0x_1 + 1 - 2x_2 $$

Agrupando los términos, el problema QUBO equivalente quedaría así:
*   **Minimizar:** $-2x_0x_1 + x_0 + x_1 - 2x_2 + \frac{1}{2}$
*   **Sujeto a:** $x_j \in \{0, 1\}$ para $j = 0, 1, 2$.

*Nota: El término independiente ($+\frac{1}{2}$) no afecta a qué variables $x_j$ son las que alcanzan el mínimo óptimo, por lo que a menudo se suele ignorar durante el proceso de optimización para simplificar las ecuaciones, aunque debemos sumarlo al final si queremos recuperar el valor de la energía original*.


## 3.4 Problemas de optimización combinatoria con el modelo QUBO

En esta sección final del capítulo, vamos a introducir técnicas que nos permitirán tomar problemas clásicos de optimización combinatoria y reescribirlos como instancias del modelo QUBO y de Ising. Aprender a formular los problemas de esta manera es el paso esencial antes de poder introducirlos en un ordenador cuántico.

### 3.4.1 Programación lineal binaria (Binary linear programming)

Los problemas de programación lineal binaria consisten en optimizar (minimizar o maximizar) una función lineal de variables binarias sujeta a una serie de restricciones lineales. 

Para ilustrar cómo transformar estos problemas a QUBO, utilizaremos el siguiente ejemplo:
*   **Minimizar:** $-5x_0 + 3x_1 - 2x_2$
*   **Sujeto a las restricciones:** 
    $x_0 + x_2 \le 1$
    $3x_0 - x_1 + 3x_2 \le 4$
*   **Donde:** $x_j \in \{0, 1\}$ para $j = 0, 1, 2$.

Como el formato QUBO significa optimización cuadrática binaria **sin restricciones (Unconstrained)**, nuestro objetivo será deshacernos de las inecuaciones ($\le$) e introducirlas dentro de la propia ecuación de energía a minimizar.


#### Paso 1: Introducir variables de holgura (Slack variables)

El primer paso es transformar las desigualdades ($\le$) en igualdades matemáticas ($=$). Para ello, sumaremos nuevas variables binarias auxiliares llamadas **variables de holgura**. Estas actuarán como comodines para que el lado izquierdo de la ecuación alcance el límite estipulado en el derecho.

*   **Para la primera restricción ($x_0 + x_2 \le 1$):** El valor mínimo posible de la izquierda es $0$. Para que llegue a sumar como máximo $1$, solo necesitamos añadir una nueva variable binaria $y_0$. La restricción se convierte en la igualdad: $x_0 + x_2 + y_0 = 1$.
*   **Para la segunda restricción ($3x_0 - x_1 + 3x_2 \le 4$):** El valor mínimo a la izquierda es $-1$ (cuando $x_1=1$ y el resto es $0$). Para llegar al máximo de $4$, necesitamos poder sumar un número que cubra una distancia de hasta $5$. Como para poder sumar hasta $5$ necesitamos tres bits binarios, introducimos tres variables de holgura ($y_1, y_2, y_3$) con sus correspondientes coeficientes: $3x_0 - x_1 + 3x_2 + y_1 + 2y_2 + 2y_3 = 4$.

*Nota general: En este tipo de problemas las variables con coeficientes positivos tenderán a ser puestas a $0$ por el optimizador para minimizar, y aquellas con coeficientes negativos a $1$.*


#### Paso 2: Crear los términos de penalización (Penalty terms)

Una vez que todas nuestras restricciones son igualdades exactas, el último paso para formular nuestro problema QUBO es convertirlas en **términos de penalización**.

La técnica es muy sencilla: reescribimos la igualdad para que esté igualada a cero (pasando el número de la derecha restando hacia la izquierda) y **elevamos la expresión resultante al cuadrado**. De esta forma, si la restricción se cumple, el cuadrado valdrá $0$, pero si no se cumple, valdrá un número estrictamente positivo.

Multiplicamos cada uno de estos cuadrados por una constante muy grande (por ejemplo $M_1$ y $M_2$) y se los sumamos a la función original que queríamos minimizar. En nuestro ejemplo, la función a minimizar completa sería:

$$ E = (-5x_0 + 3x_1 - 2x_2) + M_1(x_0 + x_2 + y_0 - 1)^2 + M_2(3x_0 - x_1 + 3x_2 + y_1 + 2y_2 + 2y_3 - 4)^2 $$

Si el ordenador cuántico encuentra el estado de mínima energía, sabremos con certeza que estará cumpliendo las restricciones, porque de lo contrario las constantes $M$ penalizarían fuertemente el resultado. Si desarrollamos todos los productos notables de los cuadrados, obtendremos una función polinómica de grado 2 con multiplicaciones como $x_0 y_1$. ¡Esa es exactamente la definición de un problema QUBO!


### 3.4.2 El problema de la mochila (The Knapsack problem)

En este famoso problema de optimización, se nos proporciona una lista de objetos $j = 0, \dots, m$, cada uno con un peso asignado $w_j$ y un valor $c_j$. Además, disponemos de una mochila con una capacidad de peso máximo $W$.

El objetivo logístico es sencillo: debemos **maximizar el valor total** de los objetos que decidimos meter en la mochila, asegurándonos de que la suma de sus pesos no exceda el límite máximo permitido $W$.


#### Formulación matemática

Podemos formular este escenario como un problema de programación lineal binaria. Para ello, definiremos una variable binaria $x_j$ para cada objeto $j$ de nuestra lista. La regla es simple:
* $x_j = 1$ si metemos el objeto en la mochila.
* $x_j = 0$ si lo dejamos fuera.

De esta forma, la formulación matemática exacta del problema queda así:
*   **Maximizar:** $\sum_j c_j x_j$
*   **Sujeto a la restricción:** $\sum_j w_j x_j \le W$
*   **Donde:** $x_j \in \{0, 1\}$ para todo $j = 0, 1, \dots, m$.


#### Transformación al modelo QUBO y variaciones

Para resolver este problema con un algoritmo cuántico, necesitamos convertirlo a un formato QUBO (sin restricciones). El proceso es idéntico al que vimos en la programación lineal general: introducimos **variables de holgura** para convertir la inecuación ($\le$) de los pesos en una igualdad matemática ($=$), y luego la incorporamos a la función objetivo principal como un término de penalización elevado al cuadrado.

**Variante con múltiples copias:**
Existe una versión del problema de la mochila en la que se nos permite elegir varias copias de un mismo objeto. Para formular esto, en lugar de usar variables binarias de $0$ y $1$, usaríamos **variables enteras** que representen el número exacto de copias que cogemos. 

Sin embargo, dado que el peso máximo de la mochila ($W$) está fijado, el número máximo de copias que podemos coger de cualquier objeto está acotado. Por tanto, siempre podemos reemplazar esas variables enteras acotadas por un conjunto de variables puramente binarias para poder seguir utilizando la formulación QUBO y los ordenadores cuánticos.

### 3.4.3 Coloreado de grafos (Graph coloring)

En el problema del coloreado de grafos, se nos da un grafo y se nos pide asignar un color a cada vértice de tal manera que **dos vértices adyacentes (conectados por una arista) nunca tengan el mismo color**. 

Normalmente, el objetivo es colorear el grafo usando el menor número posible de colores, o comprobar si es posible colorearlo usando como máximo $k$ colores (a esto se le llama comprobar si el grafo es *$k$-colorable*). Al número mínimo de colores necesarios se le llama **número cromático**.

**¿Para qué sirve esto en la vida real?**
Aunque suene a juego de niños, muchas situaciones reales se modelan así. Por ejemplo, imagina que tu empresa tiene varios proyectos y debes asignar supervisores. Algunos proyectos son incompatibles por solapamiento de horarios (aristas). Encontrar el número cromático del grafo equivale a encontrar el número mínimo de supervisores que necesitas contratar para cubrir todos los proyectos sin solapamientos.


#### Complejidad y variables de decisión

Comprobar si un grafo es $2$-colorable es relativamente fácil usando algoritmos clásicos (es equivalente a comprobar si un grafo es bipartito). Sin embargo, para cualquier $k \ge 3$, el problema es **NP-difícil (NP-hard)**.

Para formular el problema de si un grafo de $m$ vértices es $k$-colorable en el formato QUBO, primero definimos las **variables binarias**:
* Asignaremos una variable $x_{j,l}$ para cada vértice $j$ y cada color $l$ (donde $l$ va de $0$ a $k-1$).
* $x_{j,l} = 1$ si al vértice $j$ se le asigna el color $l$, y $0$ en caso contrario.

**Restricciones a cumplir:**
1. **Un solo color por vértice:** La suma de todos los colores posibles para un solo vértice debe ser exactamente $1$.
2. **Vértices conectados tienen distinto color:** Si hay una arista conectando los vértices $j$ y $h$, el producto de sus variables para un mismo color $l$ debe ser siempre $0$ (es decir, $x_{j,l} \cdot x_{h,l} = 0$).

#### Transformación al modelo QUBO

Siguiendo las reglas de QUBO, convertiremos las restricciones anteriores en penalizaciones matemáticas que sumaremos en nuestra función objetivo:

*   Para la primera restricción (un color por nodo), construimos el término de penalización restando $1$ y elevando al cuadrado. 
*   Para la segunda restricción (nodos conectados), simplemente sumamos los productos de las variables. No hace falta elevar esto último al cuadrado porque, al ser variables binarias ($0$ o $1$), su producto siempre será no-negativo (mayor o igual a cero).

La función QUBO final que le pediremos minimizar al ordenador cuántico será:

$$ Minimizar: \sum_j \left( \sum_{l=0}^{k-1} x_{j,l} - 1 \right)^2 + \sum_{(j,h) \in E} \sum_{l=0}^{k-1} x_{j,l} x_{h,l} $$

Donde $E$ es el conjunto de todas las aristas del grafo. 

**Conclusión:** Si el ordenador cuántico encuentra que la solución óptima (el mínimo de esta ecuación) es **exactamente $0$**, significa que ninguna penalización ha saltado y, por tanto, **el grafo sí es $k$-colorable**. Si el mínimo es mayor que $0$, es imposible colorearlo con esa cantidad de colores.


### 3.4.4 El Problema del Viajante (The Traveling Salesperson Problem)

El Problema del Viajante (TSP) es uno de los desafíos más famosos en la optimización combinatoria. El objetivo es muy sencillo de plantear: dado un conjunto de ciudades y los costes de viajar entre ellas, debes encontrar una ruta que **visite cada ciudad exactamente una vez, regrese a la ciudad de origen** y que minimice la suma total de los costes del viaje. 

A pesar de su aparente simplicidad, el TSP es un problema **NP-difícil (NP-hard)**. De hecho, incluso determinar si existe una ruta que visite todas las ciudades con un coste menor o igual a un valor determinado es un problema NP-completo.


#### Variables de decisión

Para formular el TSP en el formato QUBO, necesitamos representar el orden en el que se visitan las ciudades. Para ello, definiremos variables binarias $x_{j,l}$ que indicarán el orden de visita:
*   $x_{j,l} = 1$ si la ciudad $j$ es visitada en la posición $l$ de nuestra ruta.
*   $x_{j,l} = 0$ en caso contrario.

Si tenemos $m$ ciudades, los índices $j$ y $l$ irán desde $0$ hasta $m-1$. De esta forma, si queremos saber si las ciudades $j$ y $k$ se visitan de forma consecutiva (es decir, usamos la arista que las une), solo tenemos que multiplicar $x_{j,l} \cdot x_{k,l+1}$. Si el resultado es $1$, significa que ambas son consecutivas y, por tanto, tendremos que sumar el coste de ese viaje $w_{jk}$.


#### Restricciones del problema

Para que una ruta sea válida, debe cumplir dos reglas lógicas inquebrantables:
1.  **Cada ciudad se visita exactamente una vez:** Para una ciudad $j$ concreta, la suma de las posiciones posibles debe ser $1$. Matemáticamente: $\sum_l x_{j,l} = 1$.
2.  **Cada posición en la ruta está ocupada por una sola ciudad:** Para una posición $l$ concreta, la suma de todas las ciudades posibles debe ser $1$. Matemáticamente: $\sum_j x_{j,l} = 1$.

Siguiendo las reglas de QUBO que ya conocemos, igualamos estas restricciones a cero, las elevamos al cuadrado y las multiplicamos por una penalización grande $A$. 

$$ Penalizaciones = A \sum_j \left( \sum_l x_{j,l} - 1 \right)^2 + A \sum_l \left( \sum_j x_{j,l} - 1 \right)^2 $$


#### Formulación QUBO final

Si al término de penalización le sumamos la función de coste total del viaje (multiplicada por una constante $B$), obtenemos la ecuación final de nuestro problema QUBO:

$$ Minimizar: B \sum_{j,k} w_{jk} \sum_l x_{j,l} x_{k,l+1} + A \sum_j \left( \sum_l x_{j,l} - 1 \right)^2 + A \sum_l \left( \sum_j x_{j,l} - 1 \right)^2 $$

*(Nota: en la primera suma, el índice $l+1$ se calcula módulo $m$, porque la ruta es un ciclo que vuelve al inicio)*.

**Un detalle crucial:** Para que el optimizador no haga trampas y se salte las reglas para ahorrar dinero, **la penalización $A$ debe ser mucho mayor que $B$**. Si elegimos un valor de $A$ que sea mayor que el coste de cualquier ruta válida (por ejemplo $1 + m \sum w_{jk}$ y $B=1$), nos aseguramos de que cualquier solución que rompa las reglas quede totalmente descartada.


## Capítulo 4: Computación Cuántica Adiabática y Quantum Annealing

### 4.1 Computación cuántica adiabática (Adiabatic quantum computing)

En los capítulos anteriores, hemos trabajado con el modelo de circuitos cuánticos, donde el estado del sistema cambia a saltos discretos al aplicar puertas cuánticas. En la computación cuántica adiabática, por el contrario, la evolución del sistema es continua.

Esta evolución continua se rige por la famosa **ecuación de Schrödinger**:

$$ H(t) |\psi(t)\rangle = i\hbar \frac{\partial}{\partial t} |\psi(t)\rangle $$

Donde $H(t)$ es el **Hamiltoniano dependiente del tiempo**, $|\psi(t)\rangle$ es el vector de estado del sistema, $i$ es la unidad imaginaria ($\sqrt{-1}$) y $\hbar$ es la constante de Planck reducida. A diferencia de los casos anteriores donde la energía se conservaba inalterada, aquí consideramos situaciones donde la energía varía con el tiempo, por ejemplo al cambiar la intensidad o frecuencia de un pulso electromagnético que aplicamos a nuestros qubits.


#### El proceso de evolución adiabática

El segundo ingrediente para este nuevo modelo de computación es la idea de la **evolución adiabática**. A grandes rasgos, un proceso adiabático es aquel en el que la configuración de energía del sistema cambia de forma muy suave y controlada.

La observación clave de este modelo es la siguiente: si comenzamos con nuestro sistema en el **estado fundamental** (el de mínima energía) de un Hamiltoniano y lo evolucionamos adiabáticamente, sabemos que el sistema permanecerá en un estado fundamental durante todo el proceso.

Para usar esto a nuestro favor al resolver un problema de optimización, seguimos estos pasos:
1. Comenzamos con un Hamiltoniano inicial simple, $H_0$, para el cual podemos obtener y preparar fácilmente su estado fundamental.
2. Definimos un Hamiltoniano final $H_1$, cuyo estado fundamental es la solución al problema que queremos resolver (por ejemplo, el modelo Ising de nuestro problema de optimización).
3. Evolucionamos el sistema suavemente desde $H_0$ hasta $H_1$ para permanecer en el estado fundamental todo el tiempo. ¡Al final medimos el sistema y obtenemos nuestra solución!

Si ejecutamos el proceso por un tiempo total $T$, el Hamiltoniano dependiente del tiempo que consideraremos será de la forma matemática:

$$ H(t) = A(t)H_0 + B(t)H_1 $$

Aquí $A$ y $B$ son funciones reales en el intervalo $[0, T]$ tales que $A(0) = B(T) = 1$ y $A(T) = B(0) = 0$. Una elección muy común y sencilla para estas funciones es $A(t) = 1 - t/T$ y $B(t) = t/T$.


#### El Teorema Adiabático y el "Spectral Gap"

Para garantizar que el proceso sea verdaderamente adiabático (y no saltemos por error a un estado de mayor energía perdiendo nuestra solución), la evolución debe ser lo **suficientemente lenta**.

El **teorema adiabático**, probado originalmente por Max Born y Vladimir Fock, establece que el tiempo total de la evolución debe ser inversamente proporcional al cuadrado de la brecha o salto espectral (**spectral gap**). 

Este salto espectral se define como la mínima diferencia de energía que hay entre el estado fundamental y el primer estado excitado del Hamiltoniano durante toda la evolución. Mientras respetemos este límite de tiempo, tendremos garantías matemáticas de alcanzar la solución óptima.


### 4.2 Quantum annealing (Recocido cuántico)

El *quantum annealing* es la implementación práctica (y restringida) de la computación cuántica adiabática. Utiliza la misma idea central: evolucionar un sistema desde un Hamiltoniano inicial $H_0$ hasta un Hamiltoniano final $H_1$. 

Sin embargo, a diferencia del modelo adiabático general, en el *quantum annealing* estamos limitados a usar una forma matemática muy concreta:
*   **Hamiltoniano final ($H_1$):** Tiene que ser un modelo de Ising clásico, con la forma $-\sum J_{jk}Z_jZ_k - \sum h_jZ_j$.
*   **Hamiltoniano inicial ($H_0$):** Se define como $H_0 = -\sum_{j=0}^{n-1} X_j$. La magia de este Hamiltoniano es que su estado fundamental (el de mínima energía) es simplemente el estado donde todos los qubits están en superposición uniforme: $|+\rangle^{\otimes n}$, el cual es sumamente fácil de preparar aplicando puertas Hadamard a todos los qubits.


#### La evolución del sistema y la universalidad

La evolución completa del sistema a lo largo del tiempo $t$ se describe combinando ambos Hamiltonianos con las funciones $A(t)$ y $B(t)$:

$$ H(t) = -A(t) \sum_{j=0}^{n-1} X_j - B(t) \left( \sum_{j,k} J_{jk}Z_jZ_k + \sum_j h_jZ_j \right) $$

Debido a esta severa restricción en los Hamiltonianos que podemos utilizar, el *quantum annealing* **no es universal**. Esto significa que no puede ejecutar cualquier algoritmo cuántico general (como el de Shor o el de Grover), pero a cambio, nos proporciona una arquitectura de hardware que es mucho más fácil de escalar y que está perfectamente diseñada para resolver problemas de optimización combinatoria.


In [24]:
import dimod
import neal

# 1. Definimos las conexiones del problema Max-Cut para un triángulo (nodos 0, 1 y 2)
J = {(0,1): 1, (0,2): 1, (1,2): 1}
h = {}

# 2. Creamos el modelo de Ising. Le indicamos que las variables son espines (1 y -1)
triangle = dimod.BinaryQuadraticModel(h, J, 0.0, dimod.SPIN)
print("Modelo de Ising a resolver:\n", triangle)

# 3. Resolvemos usando el simulador local (simulando el quantum annealing)
# (Nota: Si en el futuro consigues el API Token de D-Wave, solo tienes que 
# cambiar neal.SimulatedAnnealingSampler() por EmbeddingComposite(DWaveSampler()))
sampler = neal.SimulatedAnnealingSampler()
result = sampler.sample(triangle, num_reads=10)

print("\nResultados obtenidos:")
print(result.aggregate())

Modelo de Ising a resolver:
 BinaryQuadraticModel({0: 0.0, 1: 0.0, 2: 0.0}, {(1, 0): 1.0, (2, 0): 1.0, (2, 1): 1.0}, 0.0, 'SPIN')

Resultados obtenidos:
   0  1  2 energy num_oc.
0 -1 -1 +1   -1.0       1
1 +1 +1 -1   -1.0       6
2 -1 +1 +1   -1.0       3
['SPIN', 3 rows, 10 samples, 3 variables]
